# `py_trees`로 로봇 파츠/여러 로봇의 동작 순서 조율하기 (탐색용)

**아직 `src/n2o`에 통합되지 않은 탐색/학습 노트북입니다.** [`py_trees`](https://github.com/splintered-reality/py_trees)
(behavior tree 라이브러리)의 기본 사용법을, n2o의 실제 `Part` 구현체(`SO101Arm`/`AmazingHand`)를 대상으로 익힙니다.
이 노트북에서 `Robot.router()`/`N2O.run()`은 전혀 바뀌지 않습니다 -- `py_trees`를 그 자리에 끼워 넣을지는 이후에
결정합니다.

## 왜 스케줄러/플래너가 필요한가

뇌파 신호 자체가 시간에 따라 순차적으로 들어오는 것은 이미 `N2O.run()`의 cycle 루프(`signal.read()` → `decoder()`
→ 정착 대기, 한 번에 하나씩)가 처리하고 있는 문제라서, 그 자체는 새 스케줄러가 필요한 이유가 아닙니다. 실제 빈
자리는 **디코딩된 명령 하나를 여러 파츠(또는 여러 로봇)에 걸쳐 어떤 순서/조건으로 실행할지**입니다 -- 지금
`Robot.router()`(`src/n2o/robot/__init__.py`)는 명령에 담긴 모든 파츠를 한 번에 스레드로 병렬 디스패치만 할 뿐,
"팔이 먼저 특정 위치에 도달한 다음에만 손을 쥐게 한다"거나 "실패하면 대체 제스처를 시도한다"거나 "스테이션 A가
끝난 다음 스테이션 B를 움직인다" 같은 순서/분기를 표현할 방법이 없습니다.

`Part.done_event`(`src/n2o/robot/part.py`)는 이미 이런 "future cross-part coordinator"를 염두에 두고 만들어진
훅입니다 -- 지금은 `Robot.router()`만 세팅/클리어하지만, docstring이 예고한 대로 다른 코드가 `done_event.wait()`/
`.is_set()`으로 파츠 하나의 완료를 기다릴 수 있게 되어 있습니다. 이 노트북 4절에서 `py_trees`의 커스텀
`Behaviour`가 바로 그 훅을 폴링하는 예시를 만들어 봅니다.

## 설치

`py_trees`는 이 노트북을 위해 `examples` 의존성 그룹에 새로 추가되었습니다 (`pyproject.toml`, `uv add --group
examples py_trees`) -- `mujoco`/`lerobot`처럼 노트북 전용 패키지라 core `dependencies`엔 들어가지 않습니다.

```bash
uv sync --group examples
uv run --group examples jupyter lab examples/10_py_trees_task_coordination.ipynb
```

In [1]:
import threading
import time

import py_trees

from n2o.robot import Robot
from n2o.robot.arm.so101 import SO101Arm
from n2o.robot.hand.amazing_hand import AmazingHand

# 실물 하드웨어에 연결하지 않습니다 -- port를 비워두면 `_real_arm`/`_servo`가 `None`으로 남고,
# 이 노트북에서 쓰는 `goal()`/`done_event`는 둘 다 I/O를 하지 않으므로 안전합니다.
arm = SO101Arm()
hand = AmazingHand()

## 1. `py_trees` 기본 개념 -- `Behaviour`와 `Status`

`py_trees`의 모든 노드는 `py_trees.behaviour.Behaviour`를 상속하고 `update()`를 오버라이드합니다. `update()`는
`py_trees.common.Status.SUCCESS`/`FAILURE`/`RUNNING` 중 하나를 반환해야 합니다 -- 트리를 한 번 "틱(tick)"할 때마다
루트부터 이 상태들이 전파됩니다. 가장 단순한 예시부터 봅니다.

In [2]:
class Hello(py_trees.behaviour.Behaviour):
    def update(self):
        print(f"  [{self.name}] tick")
        return py_trees.common.Status.SUCCESS


hello = Hello(name="hello")
hello.tick_once()
print("status:", hello.status)

  [hello] tick
status: Status.SUCCESS


## 2. `Sequence`로 순서 지정 -- 실제 `SO101Arm`/`AmazingHand`의 `goal()` 사용

`Sequence`는 자식을 왼쪽부터 순서대로 틱합니다 -- 하나가 `FAILURE`면 그 자리에서 멈추고, 모두 `SUCCESS`여야
`Sequence` 자신도 `SUCCESS`가 됩니다. 아래 `MoveTo`는 `Part.goal(cmd)`(순수 계산, I/O 없음 -- `move()`가 아닙니다)를
호출해 그 제스처의 목표 값을 조회하고, 알 수 없는 제스처면 `FAILURE`를 돌려줍니다.

In [3]:
class MoveTo(py_trees.behaviour.Behaviour):
    """`part.goal(cmd)`를 조회하는 behaviour -- I/O 없이 목표값만 계산하므로 실물 하드웨어 없이도
    안전하게 틱할 수 있습니다. 실제로 움직이려면 `goal` 대신 `move`를 부르면 되지만(§4 참고), 그건
    시간이 걸리는 동작이라 `RUNNING`을 다뤄야 합니다."""

    def __init__(self, name, part, cmd):
        super().__init__(name)
        self.part = part
        self.cmd = cmd

    def update(self):
        try:
            target = self.part.goal(self.cmd)
        except ValueError as exc:
            print(f"  [{self.name}] FAILURE ({exc})")
            return py_trees.common.Status.FAILURE
        print(f"  [{self.name}] SUCCESS -> target={target}")
        return py_trees.common.Status.SUCCESS


routine = py_trees.composites.Sequence(
    name="팔 올리고 손 쥐기",
    memory=True,
    children=[
        MoveTo("arm: up", arm, "up"),
        MoveTo("hand: grip", hand, "grip"),
    ],
)
py_trees.trees.BehaviourTree(routine).tick()
print("\nrouting status:", routine.status)
print(py_trees.display.unicode_tree(routine))

  [arm: up] SUCCESS -> target={'shoulder_pan': 87.6, 'shoulder_lift': -23.25, 'elbow_flex': 128.79, 'wrist_flex': -85.58, 'wrist_roll': 38.11}
  [hand: grip] SUCCESS -> target=[1.4, 0.0, 1.4, 0.0, 1.4, 0.0, 1.4, 0.0]

routing status: Status.SUCCESS
{-} 팔 올리고 손 쥐기
    --> arm: up
    --> hand: grip



## 3. `Selector`로 대체 동작(fallback) 지정

`Selector`는 반대로, 자식을 왼쪽부터 틱하다 하나가 `SUCCESS`면 그 자리에서 멈춥니다 -- "우선 이걸 시도하고, 실패하면
저걸 시도한다"는 폴백 로직입니다. 여기서는 존재하지 않는 손 제스처를 먼저 시도해 일부러 `FAILURE`를 내고,
`Selector`가 두 번째 자식(`"release"`, 실제 `GESTURES`에 있는 제스처)으로 넘어가는 것을 봅니다.

In [4]:
fallback = py_trees.composites.Selector(
    name="손 제스처 시도 (없으면 release로 대체)",
    memory=False,
    children=[
        MoveTo("try: 정의되지 않은 제스처", hand, "clap"),
        MoveTo("fallback: release", hand, "release"),
    ],
)
fallback.tick_once()
print("\nfallback status:", fallback.status)
print(py_trees.display.unicode_tree(fallback))

  [try: 정의되지 않은 제스처] FAILURE (unknown AmazingHand gesture: 'clap')
  [fallback: release] SUCCESS -> target=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

fallback status: Status.SUCCESS
[o] 손 제스처 시도 (없으면 release로 대체)
    --> try: 정의되지 않은 제스처
    --> fallback: release



## 4. `Parallel` + `Part.done_event`로 여러 파츠를 동시에, 비동기로 조율

지금까지는 `goal()`만 썼기 때문에 매 틱이 즉시 끝났습니다. 실물 `move()`처럼 시간이 걸리는 동작을 표현하려면
`update()`가 아직 끝나지 않았을 때 `RUNNING`을 반환해야 합니다 -- `py_trees`는 다음 틱에 같은 자리부터 다시
`update()`를 부릅니다.

`Robot.router()`는 이미 각 파츠를 스레드로 동시에 돌리고, `Part.done_event`를 그 파츠가 끝나면 set합니다
(`src/n2o/robot/__init__.py`의 `_dispatch()`). 아래 `MoveAsync`는 그 `done_event`를 그대로 재사용합니다 -- 백그라운드
스레드에서 `duration_s`초 후 `done_event.set()`을 흉내내고, `update()`는 그게 set될 때까지 `RUNNING`을 반환합니다.
`Parallel`(정책 `SuccessOnAll`)로 감싸면 팔과 손이 동시에 움직이되, 트리 쪽에서 "둘 다 끝났는지"를 명시적으로
기다릴 수 있습니다.

In [5]:
class MoveAsync(py_trees.behaviour.Behaviour):
    """`part.done_event`를 폴링하는 behaviour. `initialise()`(이 behaviour가 새로 RUNNING에 진입할 때
    한 번만 호출됨)에서 `done_event`를 클리어하고, `duration_s`초 후 set하는 백그라운드 스레드를
    띄웁니다 -- 실물 `move()`가 걸리는 시간을 흉내낸 것일 뿐, 실제 시리얼 I/O는 없습니다."""

    def __init__(self, name, part, duration_s):
        super().__init__(name)
        self.part = part
        self.duration_s = duration_s

    def initialise(self):
        self.part.done_event.clear()
        threading.Thread(
            target=lambda: (time.sleep(self.duration_s), self.part.done_event.set()),
            daemon=True,
        ).start()

    def update(self):
        if self.part.done_event.is_set():
            print(f"  [{self.name}] SUCCESS (동작 완료)")
            return py_trees.common.Status.SUCCESS
        print(f"  [{self.name}] RUNNING (아직 이동 중)")
        return py_trees.common.Status.RUNNING


parallel = py_trees.composites.Parallel(
    name="팔 + 손 동시에 움직이기",
    policy=py_trees.common.ParallelPolicy.SuccessOnAll(),
    children=[
        MoveAsync("arm: 이동 중 (0.6s)", arm, duration_s=0.6),
        MoveAsync("hand: 이동 중 (0.3s)", hand, duration_s=0.3),
    ],
)
tree = py_trees.trees.BehaviourTree(parallel)
while parallel.status != py_trees.common.Status.SUCCESS:
    tree.tick()
    time.sleep(0.1)
print("\nparallel status:", parallel.status)

  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)
  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)
  [hand: 이동 중 (0.3s)] SUCCESS (동작 완료)
  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] RUNNING (아직 이동 중)


  [arm: 이동 중 (0.6s)] SUCCESS (동작 완료)

parallel status: Status.SUCCESS


## 5. 여러 대 로봇의 순서 지정

`Robot`은 arm/hand/camera를 묶는 평범한 컨테이너라서, 물리적으로 분리된 스테이션이 여러 대면 `Robot()` 인스턴스를
여러 개 만들면 됩니다. 지금 `Robot.router()`는 인스턴스 하나 안의 파츠만 다루고, 인스턴스 여러 개 사이의 순서는
아무 데도 정의돼 있지 않습니다 -- 여기서는 스테이션 A와 스테이션 B를 각각 `Robot()`으로 만들고, `Sequence` 안에
`Parallel` 두 개(스테이션 하나당 하나, §4의 `MoveAsync` 재사용)를 넣어서 "A가 완전히 끝난 다음에만 B가 시작"하는
순서를 표현합니다.

In [6]:
station_a = Robot()
station_a.arm = SO101Arm()
station_a.hand = AmazingHand()

station_b = Robot()
station_b.arm = SO101Arm()
station_b.hand = AmazingHand()


def station_routine(name, station, arm_cmd, hand_cmd):
    return py_trees.composites.Parallel(
        name=name,
        policy=py_trees.common.ParallelPolicy.SuccessOnAll(),
        children=[
            MoveAsync(f"{name}: arm {arm_cmd}", station.arm, duration_s=0.4),
            MoveAsync(f"{name}: hand {hand_cmd}", station.hand, duration_s=0.2),
        ],
    )


multi_robot_plan = py_trees.composites.Sequence(
    name="스테이션 A 먼저, 그 다음 스테이션 B",
    memory=True,
    children=[
        station_routine("station A", station_a, "up", "grip"),
        station_routine("station B", station_b, "down", "release"),
    ],
)

tree = py_trees.trees.BehaviourTree(multi_robot_plan)
while multi_robot_plan.status != py_trees.common.Status.SUCCESS:
    tree.tick()
    time.sleep(0.1)
print("\n" + py_trees.display.unicode_tree(multi_robot_plan))

  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] RUNNING (아직 이동 중)


  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] RUNNING (아직 이동 중)
  [station A: arm up] RUNNING (아직 이동 중)
  [station A: hand grip] SUCCESS (동작 완료)


  [station A: arm up] RUNNING (아직 이동 중)


  [station A: arm up] SUCCESS (동작 완료)
  [station B: arm down] RUNNING (아직 이동 중)
  [station B: hand release] RUNNING (아직 이동 중)


  [station B: arm down] RUNNING (아직 이동 중)
  [station B: hand release] RUNNING (아직 이동 중)


  [station B: arm down] RUNNING (아직 이동 중)
  [station B: hand release] SUCCESS (동작 완료)
  [station B: arm down] RUNNING (아직 이동 중)


  [station B: arm down] SUCCESS (동작 완료)



{-} 스테이션 A 먼저, 그 다음 스테이션 B
    /_/ station A
        --> station A: arm up
        --> station A: hand grip
    /_/ station B
        --> station B: arm down
        --> station B: hand release



## 정리

`Sequence`(순서) / `Selector`(대체) / `Parallel`(동시 진행 + 완료 대기)만으로 지금 `Robot.router()`가 표현하지
못하는 세 가지 -- 파츠 간 순서, 실패 시 대체 동작, 여러 `Robot` 인스턴스 간 순서 -- 를 모두 나타낼 수 있었습니다.
`Part.done_event`를 그대로 재사용했다는 점도 중요합니다 -- 새 동기화 메커니즘을 만들 필요 없이, 기존 훅에
`py_trees`의 `RUNNING` 폴링 모델을 얹기만 하면 됐습니다.

**이 노트북이 하지 않은 것:** `Command.translate()`의 출력(`{"type", "arm", "hand"}`)을 실제로 `py_trees` 트리로
변환하는 부분, 그리고 그 트리를 `Robot.router()`/`N2O.run()`에 실제로 연결하는 부분은 아직 없습니다 -- 이건 다음
단계로, `ROADMAP.md`가 예고한 "future cross-part coordinator" 자리에 `py_trees`가 맞는 선택인지 확인하고 나서
설계할 문제입니다.